## OpenAI 연동 기본 예제

- `ChatOpenAI`로 OpenAI API에 연결하고, Prompt → LLM → OutputParser를 LCEL로 연결합니다.
- 사전 준비: `.env` 파일에 `OPENAI_API_KEY` 설정, `uv add langchain-openai python-dotenv`

In [1]:
import os
from dotenv import load_dotenv

#load_dotenv(dotenv_path='.env')
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 값은 출력하지 않고 설정 여부만 확인
if not OPENAI_API_KEY:
    raise RuntimeError(".env 파일에 OPENAI_API_KEY 가 없습니다.")
print("OPENAI_API_KEY 설정 확인 완료")

OPENAI_API_KEY 설정 확인 완료


#### OpenAI API로 호출 가능한 모델 목록 확인

- `openai` SDK(`langchain-openai`가 내부적으로 사용)의 `client.models.list()`로 현재 API 키로 사용할 수 있는 모델을 조회합니다.
- 목록 조회는 토큰을 사용하지 않아 비용이 발생하지 않습니다.
- 계정/조직에 따라 목록이 다를 수 있으며, 채팅 모델 외에 임베딩, 이미지, 음성 모델도 함께 나옵니다.

In [2]:
# openai SDK 불러오기 (langchain-openai가 내부적으로 사용하는 SDK)
from openai import OpenAI

# API 키로 클라이언트 생성 (OPENAI_API_KEY는 load_dotenv()로 .env에서 읽어 둔 값)
client = OpenAI(api_key=OPENAI_API_KEY)

# 호출 가능한 모델 목록 조회 (id 기준 오름차순 정렬)
models = sorted(client.models.list().data, key=lambda m: m.id)
print(f"조회된 모델 수: {len(models)}\n")
for m in models:
    print(f"{m.id:<45} {m.owned_by}")

조회된 모델 수: 135

babbage-002                                   system
chat-latest                                   system
chatgpt-image-latest                          system
davinci-002                                   system
gpt-3.5-turbo                                 openai
gpt-3.5-turbo-0125                            system
gpt-3.5-turbo-1106                            system
gpt-3.5-turbo-16k                             openai-internal
gpt-3.5-turbo-instruct                        system
gpt-3.5-turbo-instruct-0914                   system
gpt-4                                         openai
gpt-4-0613                                    openai
gpt-4-turbo                                   system
gpt-4-turbo-2024-04-09                        system
gpt-4.1                                       system
gpt-4.1-2025-04-14                            system
gpt-4.1-mini                                  system
gpt-4.1-mini-2025-04-14                       system
gpt-4.1-nano          

In [3]:
from langchain_core.prompts import ChatPromptTemplate

# prompt
prompt = ChatPromptTemplate.from_messages(
    [ ("system", "당신은 개발자입니다.") , 
     ("user", "{input}") ]
)
print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 개발자입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]


In [4]:
prompt_text = prompt.format(input="파이썬은 무엇인가요? 자세하게 설명해주세요")
print(type(prompt_text))
print(prompt_text)

<class 'str'>
System: 당신은 개발자입니다.
Human: 파이썬은 무엇인가요? 자세하게 설명해주세요


In [5]:
from langchain_openai import ChatOpenAI

# 모델명은 상수로 분리
OPENAI_MODEL = "gpt-4o-mini"

# OpenAI API를 사용하는 ChatOpenAI 인스턴스 생성 (base_url 생략 시 OpenAI 기본 엔드포인트 사용)
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    temperature=0.7
)
print(type(llm))
print(llm.model_name)

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
gpt-4o-mini


In [6]:
try:
    response = llm.invoke(prompt_text)
    print(type(response))
    print("응답:", response.content)
except Exception as e:
    print(f"오류 발생: {e}")

<class 'langchain_core.messages.ai.AIMessage'>
응답: 파이썬(Python)은 고수준의 프로그래밍 언어로, 1991년 귀도 반 로썸(Guido van Rossum)에 의해 처음 개발되었습니다. 파이썬은 간결하고 읽기 쉬운 문법을 가지고 있어, 프로그래밍 초보자부터 전문가까지 널리 사용되고 있습니다. 다음은 파이썬의 주요 특징과 장점에 대한 자세한 설명입니다.

### 1. **간결하고 읽기 쉬운 문법**
   - 파이썬은 코드가 자연어에 가까운 형태로 작성되기 때문에, 다른 프로그래밍 언어에 비해 가독성이 높습니다. 이는 코드 유지보수를 쉽게 하고, 협업에도 유리합니다.

### 2. **다양한 용도**
   - 파이썬은 웹 개발, 데이터 분석, 인공지능, 머신러닝, 자동화 스크립트, 게임 개발 등 다양한 분야에서 사용됩니다. 여러 라이브러리와 프레임워크가 있어, 특정 분야에 특화된 기능을 쉽게 구현할 수 있습니다.
     - 예: Django(웹 개발), NumPy(과학 계산), Pandas(데이터 분석), TensorFlow(머신러닝)

### 3. **인터프리터 언어**
   - 파이썬은 인터프리터 언어로, 코드를 한 줄씩 실행할 수 있습니다. 이는 개발자가 코드를 빠르게 테스트하고 디버깅할 수 있게 해줍니다.

### 4. **객체 지향 프로그래밍 지원**
   - 파이썬은 객체 지향 프로그래밍(OOP)을 지원하여, 코드의 재사용성과 모듈화를 촉진합니다. 클래스와 객체를 사용하여 복잡한 문제를 구조적으로 해결할 수 있습니다.

### 5. **광범위한 라이브러리와 생태계**
   - 파이썬은 다양한 서드파티 라이브러리와 패키지를 제공하는 풍부한 생태계를 가지고 있습니다. 이를 통해 개발자는 복잡한 기능을 쉽게 구현할 수 있습니다.

### 6. **커뮤니티와 지원**
   - 파이썬은 전 세계적으로 큰 커뮤니티를 가지고 있으며, 많은 자료와 튜토리얼, 문서가 제공됩니다. 문제 해결을 위한 다양한 포럼과 Q&A 사이트

## LCEL(LangChain Expression Language)
### Prompt + LLM 모델 + OutputParser

In [7]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [8]:
chain = prompt | llm | output_parser
print(type(chain))
print(chain)

<class 'langchain_core.runnables.base.RunnableSequence'>
first=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 개발자입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]) middle=[ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 't

In [9]:
response = chain.invoke({"input":"LangChain은 무엇인가요?"})

In [10]:
print(type(response))
print(response)

<class 'langchain_core.messages.base.TextAccessor'>
LangChain은 자연어 처리(NLP)와 관련된 애플리케이션을 개발하기 위한 프레임워크입니다. 특히 대규모 언어 모델(LLM)과 상호 작용하는 애플리케이션을 만들기 위해 설계되었습니다. LangChain은 다음과 같은 기능을 제공합니다:

1. **체인**: 여러 개의 모델이나 데이터 소스를 연결하여 복잡한 작업을 수행할 수 있는 체인을 구성할 수 있습니다. 예를 들어, 질문-답변 시스템에서 질문을 받고, 이를 처리한 후 답변을 생성하는 과정을 체인으로 구성할 수 있습니다.

2. **프롬프트 관리**: 프롬프트 템플릿을 쉽게 관리하고 사용할 수 있는 기능을 제공합니다. 이는 자연어 모델이 더 효과적으로 작동하도록 도와줍니다.

3. **데이터 소스 통합**: 외부 데이터베이스나 API와 통합할 수 있는 기능을 통해, 모델이 더 풍부한 정보를 바탕으로 응답을 생성할 수 있도록 합니다.

4. **기타 유틸리티**: 다양한 유틸리티와 도구들을 제공하여 개발자가 자연어 처리 애플리케이션을 더 쉽게 구현할 수 있도록 돕습니다.

LangChain은 이러한 기능들을 통해 개발자가 자연어 처리 기반의 다양한 애플리케이션, 예를 들어 챗봇, 질문-답변 시스템, 텍스트 생성기 등을 더욱 효율적으로 구축할 수 있게 해줍니다.
